# Agentic-Fuzzing -- Kaggle Notebook (Open-Source LLM Edition)

Runs the **full Agentic-Fuzzing** pipeline on Kaggle using an **open-source model**
(HuggingFace Hub **or** a Kaggle Model) instead of the Groq API:

> LLM (open-source) -> Hypothesis XML strategy -> C harness (ASan/UBSan) -> crash triage

| Pipeline stage | What happens |
|---|---|
| LLM (open-source) | Writes a `hypothesis` XML-generating strategy |
| `generator/` | AST-validates the strategy, then live-loads it |
| C harness `mxml_harness` | Parses each XML under ASan/UBSan, classifies exit code |
| Refine loop | Feeds acceptance-rate + crash signatures back to the LLM |
| `triage/` | Deduplicates, minimizes & verifies any crashes found |

### Before you run (important)
1. **Settings -> Accelerator -> GPU T4 (or A10G)** -- best quality for 7 B models.
   No GPU? Keep `MODEL_SOURCE = "hf"` and set `MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"`
   (slower, but runs on CPU).
2. **Settings -> toggle "Internet" ON** -- needed to clone the repo and (for the
   `hf` source) download weights. For `kaggle_input` (an attached Kaggle Model)
   you can turn Internet OFF.

Run every cell **top-to-bottom**. Only the **Configuration** cell needs editing.


In [ ]:
import sys, subprocess

def _need(pkg, mod):
    try:
        __import__(mod); return False
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        return True

for pkg, mod in [
    ("hypothesis", "hypothesis"),
    ("httpx", "httpx"),
    ("python-dotenv", "dotenv"),
    ("transformers", "transformers"),
    ("torch", "torch"),
    ("huggingface_hub", "huggingface_hub"),
    ("kagglehub", "kagglehub"),
]:
    print(("installed" if _need(pkg, mod) else "ok").ljust(9), pkg)

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())


### Configuration  (edit THIS cell)

Pick the model source and the fuzzing-loop parameters.

- `MODEL_SOURCE = "hf"` -> download `MODEL_NAME` from the HuggingFace Hub
  (non-gated models work on Kaggle).
- `MODEL_SOURCE = "kaggle_input"` -> read a Kaggle Model you attached via
  **Settings -> Source -> Add model**; paste its local directory (the one
  containing `config.json`) into `KAGGLE_MODEL_REF`, e.g.
  `/kaggle/input/qwen-2-5-7b-instruct`.
- `MODEL_SOURCE = "kaggle_kagglehub"` -> fetch via the `kagglehub` library from
  `KAGGLE_MODEL_REF = "owner/model-slug"` (requires Kaggle credentials).

Sensible defaults are provided; everything else is optional to change.


In [ ]:
import os

# Fixed project location (the notebook clones here; keep this path).
PROJECT = "/kaggle/working/Agentic-Fuzzing"
REPO    = "https://github.com/Faseeh24/Agentic-Fuzzing.git"

# ------------------------------------------------------------------
# EDIT THESE VALUES TO CONFIGURE YOUR RUN
# ------------------------------------------------------------------
# Source of the model: "hf" (Hub id) | "kaggle_input" (attached local dir) |
# "kaggle_kagglehub" ("owner/model/version").
MODEL_SOURCE = "kaggle_input"

# Source A: HuggingFace Hub repo id. Non-gated models work on Kaggle.
#   "Qwen/Qwen2.5-7B-Instruct"            <- default (best quality, GPU)
#   "mistralai/Mistral-7B-Instruct-v0.3"
#   "Qwen/Qwen2.5-1.5B-Instruct"          <- fast / runs on CPU
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Source B/C: Kaggle Model reference (only used if MODEL_SOURCE != "hf").
#   kaggle_input    -> local dir with config.json, e.g. /kaggle/input/qwen-2-5-7b-instruct
#   kaggle_kagglehub -> "owner/model-slug" or "owner/model-slug/version"
KAGGLE_MODEL_REF = "/kaggle/input/models/qwen-lm/qwen2.5/transformers/7b-instruct/1"

# Fuzzing-loop parameters (the LLM makes one strategy call per iteration)
MAX_ITERATIONS = 3     # seed + refine cycles
NUM_EXAMPLES   = 50   # XML inputs generated & fuzzed per iteration
WALL_CLOCK_CAP = 1800  # seconds, overall safety back-stop
# With a local model there is no per-token cost, so this disables the
# cost-based early-stop gate inside the orchestrator.
COST_BUDGET    = 1e9
# ------------------------------------------------------------------

# Resolve model source -> a value from_pretrained() accepts (Hub id OR local dir).
# The client loads lazily on the first chat() and caches the model singleton.
if MODEL_SOURCE == "hf":
    os.environ["HF_MODEL_NAME"] = MODEL_NAME
elif MODEL_SOURCE == "kaggle_input":
    assert KAGGLE_MODEL_REF, "set KAGGLE_MODEL_REF to the /kaggle/input/... dir"
    os.environ["HF_MODEL_NAME"] = KAGGLE_MODEL_REF
elif MODEL_SOURCE == "kaggle_kagglehub":
    import kagglehub
    os.environ["HF_MODEL_NAME"] = kagglehub.model_download(KAGGLE_MODEL_REF)
else:
    raise SystemExit("MODEL_SOURCE must be 'hf' | 'kaggle_input' | 'kaggle_kagglehub'")

os.environ.setdefault("PYTHONPATH", PROJECT)

print("MODEL SOURCE:", MODEL_SOURCE)
print("MODEL PATH  :", os.environ["HF_MODEL_NAME"])
print("Loop  : iterations=%d  examples/iter=%d  wall_cap=%ds" %
      (MAX_ITERATIONS, NUM_EXAMPLES, int(WALL_CLOCK_CAP)))


### Clone the repo & build the ASan/UBSan C harness

Compiles the vendored Mini-XML library + `harness/mxml_harness.c` with
AddressSanitizer + UndefinedBehaviorSanitizer (same `Makefile` as the repo).


In [ ]:
import os, subprocess, shutil

# Clone Agentic-Fuzzing
if not os.path.isdir(os.path.join(PROJECT, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", REPO, PROJECT], check=True)
else:
    subprocess.run(["git", "-C", PROJECT, "pull"], check=False)

os.chdir(PROJECT)
os.environ["PYTHONPATH"] = PROJECT
print("Repo ready at", PROJECT)

# Clone MXML target and checkout required commit
MXML = os.path.join(PROJECT, "target", "mxml")
MXML_COMMIT = "e6824d899d949387fb0156af6f4101373b9be519"

if not os.path.isdir(os.path.join(MXML, ".git")):
    subprocess.run(
        ["git", "clone", "https://github.com/michaelrsweet/mxml.git", MXML],
        check=True
    )

subprocess.run(["git", "-C", MXML, "fetch", "--all"], check=True)
subprocess.run(["git", "-C", MXML, "checkout", MXML_COMMIT], check=True)

# Generate MXML config.h
if not os.path.exists(os.path.join(MXML, "config.h")):
    subprocess.run(["./configure"], cwd=MXML, check=True)

# Build harness
r = subprocess.run(["make", "-C", "harness", "all"], capture_output=True, text=True)
if r.returncode != 0:
    print((r.stderr or "")[-2500:])
    raise SystemExit("Harness build FAILED - see output above.")

assert os.path.exists("harness/mxml_harness"), "mxml_harness binary missing!"
print("Harness built ->", os.path.abspath("harness/mxml_harness"))

### Use an open-source LLM (swap out the Groq client)

The next cell writes a local HuggingFace Transformers `LLMClient` over the
repo's `agent/llm_client.py`. It exposes the *same interface*
(`LLMClient(model=None)`, `.is_available()`, `.chat(messages, timeout)`), so
`agent/orchestrator.py` runs **unchanged**. The model comes from `HF_MODEL_NAME`
(resolved in the Configuration cell above).

Then run the patch cell and the smoke-test cell.


In [ ]:
%%writefile /kaggle/working/Agentic-Fuzzing/agent/llm_client.py
"""
agent/llm_client.py — Open-source LLM client for the Kaggle notebook.

This file OVERWRITES the repo's Groq-only client (agent/llm_client.py) so the
same orchestrator pipeline (agent/orchestrator.py) runs locally with any
HuggingFace open-source model instead of calling the Groq HTTP API.

The public interface is identical to the original Groq LLMClient, so
agent/orchestrator.py works UNCHANGED:

    LLMClient(model=None).is_available() -> bool
    LLMClient(model=None).chat(messages, timeout=120.0) -> str

Model selection
---------------
    os.environ["HF_MODEL_NAME"]  -> model repo id (set in the notebook config cell)

The model is loaded lazily on the first chat() call and caches the model singleton.
A module-level singleton cache makes the (slow) load happen only once across every LLMClient() created
during a notebook run — the smoke-test cell pre-loads it, then the pipeline reuses the cached instance.

Robustness
----------
If a model is too big for GPU memory, generation auto-retries on CPU with offloading. If loading
or generation ever fails, chat() raises RuntimeError — the orchestrator catches that exception and
degrades gracefully to the bundled known-good strategy (fuzzer/fallback_strategy.py), so the loop ALWAYS
keeps running.

Suggested (non-gated) models for Kaggle:
    Qwen/Qwen2.5-7B-Instruct          (best quality, needs a GPU T4/A10G)
    mistralai/Mistral-7B-Instruct-v0.3
    Qwen/Qwen2.5-1.5B-Instruct          (fast / works without a GPU)
"""

from __future__ import annotations

import logging
import os
import time
from typing import Optional

import torch

logger = logging.getLogger(__name__)

_DEFAULT_MODEL = "Qwen/Qwen2.5-7B-Instruct"

# Singleton cache: model name -> LLMClient instance whose model is loaded.
# Re-using the cache means the heavy download+load happens just once.
_CACHE: dict = {}


def _cuda_available() -> bool:
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False


class LLMClient:
    """Open-source LLM client backed by HuggingFace Transformers."""

    def __init__(self, model: Optional[str] = None) -> None:
        self._model_name = model or os.getenv("HF_MODEL_NAME", _DEFAULT_MODEL)
        self._tokenizer = None
        self._model = None
        self._device = "cuda" if _cuda_available() else "cpu"
        logger.info("LLMClient configured model=%s device=%s",
                     self._model_name, self._device)

    # -- model lifecycle -------------------------------------------------

    def _load(self) -> None:
        """Load (or reuse) the tokenizer + model. Idempotent and cached."""
        cached = _CACHE.get(self._model_name)
        if cached is not None and cached._model is not None:
            self._tokenizer = cached._tokenizer
            self._model = cached._model
            self._device = cached._device
            print(f"[llm_client] Reusing cached model: {self._model_name}")
            return

        from transformers import AutoTokenizer, AutoModelForCausalLM

        print(f"[llm_client] Loading open-source model: {self._model_name}  "
              f"(device={self._device})")
        t0 = time.time()
        self._tokenizer = AutoTokenizer.from_pretrained(self._model_name)
        # Some tokenizers (e.g. Qwen) have no pad token; fall back to eos.
        if self._tokenizer.pad_token is None:
            self._tokenizer.pad_token = self._tokenizer.eos_token

        weight_dtype = torch.bfloat16 if self._device == "cuda" else torch.float32
        try:
            self._model = AutoModelForCausalLM.from_pretrained(
                self._model_name,
                dtype=weight_dtype,
                device_map="auto",
            )
        except Exception as exc:
            # GPU OOM / dtype issues -> retry on CPU with offloading.
            print(f"[llm_client] GPU load failed ({exc!r}); "
                  f"retrying on CPU (float32, offloaded).")
            import gc
            gc.collect()
            try:
                self._model = AutoModelForCausalLM.from_pretrained(
                    self._model_name,
                    dtype=torch.float32,
                    device_map="auto",
                    offload_folder="offload",
                )
            except Exception as exc2:
                raise RuntimeError(
                    f"Failed to load model '{self._model_name}': {exc2!r}"
                ) from exc2

        self._model.eval()
        _CACHE[self._model_name] = self
        print(f"[llm_client] Model '{self._model_name}' loaded "
              f"in {time.time() - t0:.1f}s")

    def is_available(self) -> bool:
        """Return True.

        The local model is treated as available; any load/generation failure
        surfaces as an exception in chat(), which the orchestrator catches and
        degrades to the bundled fallback strategy. This mirrors the Groq client
        whose is_available() gate only guards against a missing API key.
        """
        return True

    # -- generation ------------------------------------------------------

    def chat(self, messages: list[dict], timeout: float = 120.0) -> str:
        """Generate a response from the open-source model.

        Parameters
        ----------
        messages : list[dict]
            OpenAI-style messages, e.g.
            [{"role": "system", "content": "..."}, {"role": "user", "content": "..."}]
        timeout : float
            Accepted for interface compatibility (a local model is not
            rate-limited); not enforced.

        Returns
        -------
        str
            The assistant's response text (everything generated after the prompt).
        """
        if self._model is None:
            self._load()
        if self._model is None or self._tokenizer is None:
            raise RuntimeError(
                f"Open-source model '{self._model_name}' is not loaded."
            )

        # Normalise to plain role/content dicts.
        msgs = [
            {"role": str(m.get("role", "user")), "content": str(m.get("content", ""))}
            for m in messages
        ]

        # Use the tokenizer's native chat template when available
        # (Qwen / Mistral / Llama-3 all ship one). Fall back to a flat concat.
        try:
            input_ids = self._tokenizer.apply_chat_template(
                msgs, return_tensors="pt", add_generation_prompt=True
            )
        except Exception:
            rendered = "\n".join(f"{m['role']}: {m['content']}" for m in msgs)
            enc = self._tokenizer(rendered, return_tensors="pt")
            if hasattr(enc, "input_ids"):
                input_ids = enc.input_ids
            elif isinstance(enc, dict):
                input_ids = enc["input_ids"]
            else:
                input_ids = enc

        # Robustly ensure input_ids is a torch.Tensor
        if not isinstance(input_ids, torch.Tensor):
            if hasattr(input_ids, "input_ids"):
                input_ids = input_ids.input_ids
            elif isinstance(input_ids, list):
                input_ids = torch.tensor(input_ids)
            else:
                try:
                    input_ids = torch.as_tensor(input_ids)
                except Exception as e:
                    raise RuntimeError(f"Failed to convert input_ids to Tensor: {e}")

        input_ids = input_ids.to(self._model.device)

        # Don't let the prompt blow the context window.
        # Use model_max_length if available, otherwise default to 4096.
        max_total = getattr(self._tokenizer, "model_max_length", 4096)
        # Ensure max_total is a reasonable number (sometimes it's inf)
        if not isinstance(max_total, (int, float)) or max_total > 100_000:
            max_total = 4096
        
        tailroom = max_total - 2048
        if input_ids.shape[-1] > tailroom:
            input_ids = input_ids[..., -tailroom:]

        with torch.no_grad():
            generated = self._model.generate(
                input_ids,
                max_new_tokens=2048,
                temperature=0.2,
                do_sample=True,
                top_p=0.95,
                repetition_penalty=1.05,
                pad_token_id=self._tokenizer.pad_token_id,
                eos_token_id=self._tokenizer.eos_token_id,
            )

        # Drop the prompt and decode only the newly generated tokens.
        prompt_len = input_ids.shape[-1]
        text = self._tokenizer.decode(
            generated[0][prompt_len:], skip_special_tokens=True
        )
        return text


def chat(messages: list[dict], timeout: float = 120.0) -> str:
    """Convenience helper — mirrors the Groq module-level chat()."""
    return LLMClient().chat(messages, timeout=timeout)


### Patch the cosmetic provider label

Relabels the `"groq"` provider tag in `agent/orchestrator.py` to `"local-llm"`
so the run summary is accurate. (Orchestrator LOGIC is not touched.)


In [ ]:
import os
op = os.path.join(PROJECT, "agent", "orchestrator.py")
src = open(op, encoding="utf-8").read()
patches = [
    ('llm_provider = "groq"', 'llm_provider = "local-llm"'),
    ("mxml (Groq)", "mxml (Local LLM)"),
    ("Groq API key not set. Set GROQ_API_KEY in .env", "open-source model unavailable"),
]
for old, new in patches:
    assert old in src, "pattern not found: " + repr(old)
    src = src.replace(old, new)
open(op, "w", encoding="utf-8").write(src)
print("Patched agent/orchestrator.py -> provider='local-llm'")


### Pre-load the model & smoke test

Downloads (first run only) and loads the model, then asks it for a tiny code
snippet. This gives fast feedback that the LLM backend works before the full
pipeline. The loaded model is cached (`_CACHE`) and reused by the pipeline cell.


In [ ]:
import os, sys
os.environ.setdefault("HF_MODEL_NAME", MODEL_NAME)
sys.path.insert(0, PROJECT)

print("Loading model from:", os.environ["HF_MODEL_NAME"])

from agent.llm_client import LLMClient

client = LLMClient()
print("LLM backend:", type(client).__module__ + "." + type(client).__name__,
      "-> model:", client._model_name)
print("is_available:", client.is_available())

resp = client.chat([
    {"role": "system", "content": "You are a terse assistant. Reply with ONLY a python code snippet, no prose."},
    {"role": "user",   "content": "Write a one-line python snippet that prints the number 42."},
], timeout=120)

print("---- model reply ----")
print(resp.strip())
looks_code = "print" in resp.lower() and "42" in resp
if resp.strip() and looks_code:
    print("Smoke test passed - model responds with code.")
else:
    print("WARNING - no code detected; full pipeline will still run (the")
    print("built-in validator + fallback strategy keep things safe).")


### Run the full pipeline

Seeds an XML strategy from the LLM, AST-validates + live-loads it, fuzzes it
through the ASan/UBSan harness, refines using acceptance-rate + crash
signatures, and finally runs crash triage (dedupe / minimize / verify).


In [ ]:
import os, sys
os.environ.setdefault("HF_MODEL_NAME", MODEL_NAME)
os.environ["PYTHONPATH"] = PROJECT
sys.path.insert(0, PROJECT)

from agent.orchestrator import run_orchestrator

result = run_orchestrator(
    max_iterations=MAX_ITERATIONS,
    num_examples=NUM_EXAMPLES,
    wall_clock_cap=WALL_CLOCK_CAP,
    cost_budget=COST_BUDGET,
    run_triage=True,
)

print("\n======== PIPELINE RESULT ========")
for k, v in result.items():
    print(f"  {k}: {v}")


### Results & artifacts

Inspect generated strategies, per-iteration classification logs, and any crash
reproducers under `triage/crashes/<signature>/`.


In [ ]:
import json
from pathlib import Path

root = Path(PROJECT)

summ = root / "fuzzer" / "logs" / "loop_summary.md"
if summ.exists():
    print("loop_summary.md:")
    print(summ.read_text())

print("\nstrategy files:")
for f in sorted((root / "fuzzer/strategies").glob("iteration_*.py")):
    print(f"  {f.name}  ({f.stat().st_size} bytes)")

print("\niteration logs:")
for f in sorted((root / "fuzzer/logs").glob("iteration_*.jsonl")):
    rec = json.loads(f.read_text(encoding="utf-8").splitlines()[0])
    r = rec["results"]
    print(f"  {f.name}: total={r['total']} accept={r['acceptance_rate']:.0%} "
          f"sanitizer={r['sanitizer']} timeout={r['timeout']} bug_crash={r['bug_crash']}")

print("\ntriage (crashes):")
cd = root / "triage" / "crashes"
if cd.exists() and any(cd.iterdir()):
    for d in sorted(cd.iterdir()):
        if not d.is_dir():
            continue
        repro = "reproducer_minimized.xml" if (d / "reproducer_minimized.xml").exists() else "reproducer.xml"
        print(f"  {d.name} -> {repro}")
        rep = d / repro
        if rep.exists():
            print("    input:", repr(rep.read_text(encoding="utf-8")[:120]))
        sr = d / "sanitizer_report.txt"
        if sr.exists():
            print("    stderr:", sr.read_text(encoding="utf-8")[:300])
else:
    print("  (no crashes found - nothing to triage)")


### Save artifacts to Output

Kaggle preserves files under `/kaggle/working/` when you **commit** the run.
To make the generated strategies, per-iteration logs and crash reproducers easy
to recover, this cell copies them into
`/kaggle/working/output/agentic_fuzzing_run/` (shown in the run's Output tab).


In [ ]:
import shutil
from pathlib import Path

out_root = Path("/kaggle/working/output/agentic_fuzzing_run")
out_root.mkdir(parents=True, exist_ok=True)
root = Path(PROJECT)

copied = []
for src_sub, dst_name in [
    ("fuzzer/strategies", "strategies"),
    ("fuzzer/logs", "logs"),
    ("triage/crashes", "triage_crashes"),
]:
    src = root / src_sub
    dst = out_root / dst_name
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        copied.append(dst_name)

ls = root / "fuzzer" / "logs" / "loop_summary.md"
if ls.exists():
    shutil.copy2(ls, out_root / "loop_summary.md")

print("Saved artifacts to", out_root)
for name in copied:
    d = out_root / name
    print(" ", name, "/", len(list(d.rglob("*"))), "entries")
for p in sorted(out_root.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(out_root), f"({p.stat().st_size} bytes)")


### Done

- To re-run with a different model: edit the **Configuration** cell (the
  `MODEL_SOURCE` / `MODEL_NAME` / `KAGGLE_MODEL_REF` line), then
  **Runtime -> Restart and run all** (or run from the smoke-test cell onward).
- Generated strategies: `fuzzer/strategies/iteration_*.py`
- Per-example logs (incl. real ASan/UBSan stderr): `fuzzer/logs/iteration_*.jsonl`
- Confirmed crash reproducers: `triage/crashes/<signature>/`
- Archived copy (strategies + logs + crashes + summary) saved to
  `/kaggle/working/output/agentic_fuzzing_run/`
